# FROST Signing Protocol

This notebook explains FROST threshold signing after key generation and preprocessing.

Prerequisites:

- [02-schnorr.ipynb](02-schnorr.ipynb): Schnorr structure and verification
- [03-keygen.ipynb](03-keygen.ipynb): long-lived secret shares $s_i$ and group key $Y$
- [04-preprocessing.ipynb](04-preprocessing.ipynb): nonce pairs and commitment lists $(D_{ij}, E_{ij})$

## Goal and Design

FROST signing is designed for unrestricted parallel signing sessions while keeping the final signature standard Schnorr.

The final signature is:

$$
\sigma = (R, z),
$$

verifiable under the group public key $Y$ with ordinary Schnorr verification.

FROST achieves this with:

- preprocessed commitment pairs per participant
- binding values that tie responses to message and signer/commitment set
- per-participant response checks by a signature aggregator

## Why Binding Values Are Needed

Without binding, an adversary can try to manipulate commitment/message combinations across concurrent sessions (the Drijvers-style setting).

FROST binds each participant's contribution to:

- participant identity
- message $m$
- ordered commitment set $B$

using:

$$
\rho_i = H_1(i, m, B).
$$

This prevents valid mixing of shares from different messages or different commitment sets.

## Signing Objects and Notation

Let:

- $S$ be the selected signing subset, with size $\alpha$ where $t \leq \alpha \leq n$
- $Y$ be the group public key
- $s_i$ be participant $i$'s long-lived secret share
- $(D_i, E_i)$ be participant $i$'s next unused preprocessed commitment pair
- $B = \langle (i, D_i, E_i) \rangle_{i \in S}$ be the ordered commitment list

From preprocessing, each participant has private nonces $(d_i, e_i)$ matching $(D_i, E_i) = (g^{d_i}, g^{e_i})$.

## Step-by-Step Signing (Single-Round Online Phase)

1. **Aggregator selects participants and commitments**

   The signature aggregator (SA) chooses signing set $S$ and fetches one unused commitment pair per signer, then constructs

   $$
   B = \langle (i, D_i, E_i) \rangle_{i \in S}.
   $$

2. **SA sends $(m, B)$ to each signer**

3. **Each signer validates input**

   Each signer checks message policy and commitment validity.

4. **Each signer computes binding values and commitments**

   $$
   \rho_i = H_1(i, m, B),
   $$

   $$
   R_i = D_i \cdot E_i^{\rho_i} = g^{d_i + e_i\rho_i}.
   $$

   Group commitment:

   $$
   R = \prod_{i \in S} R_i.
   $$

5. **Each signer computes challenge and response share**

   $$
   c = H_2(R, Y, m),
   $$

   $$
   z_i = d_i + e_i\rho_i + \lambda_i s_i c,
   $$

   where $\lambda_i$ is participant $i$'s Lagrange coefficient over signer set $S$.

6. **Signer deletes single-use nonce pair and sends $z_i$ to SA**

7. **SA verifies each response share**

   $$
   g^{z_i} \stackrel{?}{=} R_i \cdot Y_i^{c\lambda_i}.
   $$

   If a check fails, SA identifies misbehaviour and aborts.

8. **SA aggregates final response and publishes signature**

   $$
   z = \sum_{i \in S} z_i,
   $$

   $$
   \sigma = (R, z).
   $$

## Security and Operational Notes

- Nonce pairs are single-use; reuse can expose long-lived secret shares.
- SA can coordinate and validate, but malformed behaviour should cause abort + investigation.
- Preprocessing allows asynchronous readiness while keeping signing rounds low.
- Binding keeps concurrent sessions safe against share-mixing style attacks.

## How This Connects Back

- From [01-threshold-polynomials.ipynb](01-threshold-polynomials.ipynb): Lagrange coefficients and share consistency ideas.
- From [02-schnorr.ipynb](02-schnorr.ipynb): same final Schnorr verification shape.
- From [03-keygen.ipynb](03-keygen.ipynb): secret shares $s_i$ and public shares $Y_i$.
- From [04-preprocessing.ipynb](04-preprocessing.ipynb): commitment-pair generation and storage discipline.

This notebook is the step where those pieces become one full threshold signature protocol.